# RAPiDock Colab Runner for GHS-R Peptides

This notebook runs RAPiDock in Google Colab (Linux) and exports peptide docking outputs for your local project.

**Important:** RAPiDock ships precompiled Linux modules (for example `PeptideBuilder.so`) that are **not compatible with Colab's default Python 3.12** (you may see `undefined symbol: _PyUnicode_Ready`). This notebook therefore installs a **conda Python 3.10** environment and runs `inference.py` via `conda run`.

Workflow:
1. Install Miniforge + Python 3.10 conda env.
2. Clone RAPiDock and install dependencies **into that env** (torch + PyG matched).
3. Download `rapidock_local.pt` checkpoint.
4. Upload your receptor PDB and peptide CSV.
5. Run either a single smoke test or full batch.
6. Zip outputs and download them.

After download, place outputs into your local repo as `outputs_rapidock/`, then run locally:

```bash
python scripts/parse_rapidock_results.py
python scripts/parse_results.py
python scripts/visualize.py
```

In [ ]:
#@title 1) Install Miniforge + Python 3.10 env (RAPiDock `.so` requires Python ≤3.11)
import os
import subprocess
from pathlib import Path

CONDA_ENV = 'rapidock310'
os.environ['CONDA_ENV'] = CONDA_ENV

MAMBA_ROOT = Path('/usr/local/miniforge3')
CONDA = MAMBA_ROOT / 'bin' / 'conda'

if not CONDA.exists():
    installer = Path('/tmp/Miniforge3.sh')
    url = 'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh'
    subprocess.check_call(['wget', '-q', '-O', str(installer), url])
    subprocess.check_call(['bash', str(installer), '-b', '-p', str(MAMBA_ROOT)])

exists = subprocess.run(
    [str(CONDA), 'env', 'list'],
    check=True,
    capture_output=True,
    text=True,
).stdout
if f'{CONDA_ENV} ' not in exists:
    subprocess.check_call([str(CONDA), 'create', '-y', '-n', CONDA_ENV, 'python=3.10'])

print('conda env ready:', CONDA_ENV)
subprocess.check_call([str(CONDA), 'run', '-n', CONDA_ENV, 'python', '-V'])

In [ ]:
#@title 2) Clone RAPiDock
!git clone https://github.com/huifengzhao/RAPiDock.git
%cd /content/RAPiDock

In [ ]:
#@title 3) Install dependencies into conda env (torch+pyg matched)
import os
import subprocess

import torch

CONDA = '/usr/local/miniforge3/bin/conda'
CONDA_ENV = os.environ.get('CONDA_ENV', 'rapidock310')

subprocess.check_call([CONDA, 'run', '-n', CONDA_ENV, 'python', '-m', 'pip', 'install', '-U', 'pip'])

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        'pyyaml',
        'pandas',
        'biopython',
        'MDAnalysis>=2.9.0',
        'fair-esm',
        'e3nn',
        'pyrosetta-installer',
    ]
)

subprocess.check_call([CONDA, 'run', '-n', CONDA_ENV, 'python', '-m', 'pip', 'install', 'rdkit'])

ver = torch.__version__
if '+' in ver:
    torch_base, pt_cuda = ver.split('+', 1)
else:
    torch_base, pt_cuda = ver, 'cpu'

if pt_cuda == 'cpu':
    torch_index = 'https://download.pytorch.org/whl/cpu'
else:
    torch_index = f'https://download.pytorch.org/whl/{pt_cuda}'

print('Notebook torch (for CUDA wheel selection):', ver)
print('Installing conda-env torch from:', torch_index)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        f'torch=={torch_base}',
        '--index-url',
        torch_index,
    ]
)

probe = subprocess.check_output(
    [CONDA, 'run', '-n', CONDA_ENV, 'python', '-c', 'import torch; print(torch.__version__)'],
    text=True,
).strip()
if '+' in probe:
    env_base, env_cuda = probe.split('+', 1)
else:
    env_base, env_cuda = probe, 'cpu'
pyg_tag = f'cu{env_cuda.replace("cu", "")}' if env_cuda.startswith('cu') else 'cpu'
pyg_url = f'https://data.pyg.org/whl/torch-{env_base}+{pyg_tag}.html'

subprocess.run(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'uninstall',
        '-y',
        'torch-geometric',
        'torch-scatter',
        'torch-sparse',
        'torch-cluster',
        'pyg-lib',
    ],
    check=False,
)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        'pyg_lib',
        'torch_scatter',
        'torch_sparse',
        'torch_cluster',
        'torch_geometric',
        '-f',
        pyg_url,
    ]
)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-c',
        'import MDAnalysis, rdkit, torch, torch_geometric, torch_scatter, torch_sparse, torch_cluster; '
        'print("torch:", torch.__version__); '
        'print("MDAnalysis:", MDAnalysis.__version__); '
        'print("RDKit:", rdkit.__version__); '
        'print("torch_geometric:", torch_geometric.__version__)',
    ]
)

In [ ]:
#@title 4) Download RAPiDock local checkpoint
!mkdir -p /content/RAPiDock/train_models/CGTensorProductEquivariantModel
!wget -O /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt "https://zenodo.org/records/14193621/files/rapidock_local.pt?download=1"
!ls -lh /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt

In [ ]:
#@title 5) Upload inputs from your local project
# Upload these files from local repo:
# - data/ghsr_receptor.pdb  (or data/ghsr_pocket.pdb)
# - inputs_rapidock/protein_peptide.csv (for full-batch mode)
#
# Optional for single-case mode: only receptor PDB is needed.
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

In [ ]:
#@title 6) Set paths and quick sanity check
import os

CONDA_ENV = os.environ.get('CONDA_ENV', 'rapidock310')
os.environ['CONDA_ENV'] = CONDA_ENV

RECEPTOR_PDB = '/content/ghsr_receptor.pdb'  # Change if uploaded with different name
BATCH_CSV = '/content/protein_peptide.csv'   # Change if uploaded with different name
OUTPUT_SINGLE = '/content/outputs_rapidock_test'
OUTPUT_BATCH = '/content/outputs_rapidock'

for _k in ['RECEPTOR_PDB', 'BATCH_CSV', 'OUTPUT_SINGLE', 'OUTPUT_BATCH', 'CONDA_ENV']:
    os.environ[_k] = locals()[_k]

print('CONDA_ENV:', CONDA_ENV)
print('RECEPTOR_PDB exists:', os.path.exists(RECEPTOR_PDB))
print('BATCH_CSV exists:', os.path.exists(BATCH_CSV))
print('RAPiDock checkpoint exists:', os.path.exists('/content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt'))

In [ ]:
#@title 7) Single-case smoke test (hexarelin)
!conda run --no-capture-output -n "$CONDA_ENV" python /content/RAPiDock/inference.py \
  --complex_name hexarelin_test \
  --protein_description "$RECEPTOR_PDB" \
  --peptide_description "H[DTR]AW[DPN]K" \
  --output_dir "$OUTPUT_SINGLE" \
  --model_dir /content/RAPiDock/train_models/CGTensorProductEquivariantModel \
  --ckpt rapidock_local.pt \
  --scoring_function ref2015 \
  --N 1 --batch_size 1 \
  --no_final_step_noise \
  --inference_steps 8 --actual_steps 8 \
  --conformation_partial 1:1:1 \
  --cpu 4

In [ ]:
#@title 8) Verify single-case output artifacts
!ls -R "$OUTPUT_SINGLE"

# Expected key file:
# /content/outputs_rapidock_test/hexarelin_test/ref2015_score.csv

In [ ]:
#@title 9) Full peptide batch run (uses uploaded protein_peptide.csv)
# Skip this cell if you only need single-case smoke test.
!conda run --no-capture-output -n "$CONDA_ENV" python /content/RAPiDock/inference.py \
  --protein_peptide_csv "$BATCH_CSV" \
  --output_dir "$OUTPUT_BATCH" \
  --model_dir /content/RAPiDock/train_models/CGTensorProductEquivariantModel \
  --ckpt rapidock_local.pt \
  --scoring_function ref2015 \
  --N 10 --batch_size 4 \
  --no_final_step_noise \
  --inference_steps 16 --actual_steps 16 \
  --conformation_partial 1:1:1 \
  --cpu 8

In [ ]:
#@title 10) Zip and download RAPiDock outputs
from google.colab import files
import os

# Choose which output folder to export:
EXPORT_DIR = OUTPUT_BATCH if os.path.exists(OUTPUT_BATCH) else OUTPUT_SINGLE
EXPORT_ZIP = '/content/outputs_rapidock_colab.zip'

!zip -r "$EXPORT_ZIP" "$EXPORT_DIR"
files.download(EXPORT_ZIP)

## Local follow-up after download

1. Unzip into your repo so outputs land under `outputs_rapidock/`.
2. Run locally:

```bash
python scripts/parse_rapidock_results.py
python scripts/parse_results.py
python scripts/visualize.py
```

This merges RAPiDock peptide rows with Boltz small-molecule rows in `results/ranking.csv`.